   # Preparing Data for Modeling

## 1. Load and clean the data

Load the raw dataset and reapply the `TotalCharges` fix from the exploration notebook (blank values for new customers set to 0).

In [1]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = df["TotalCharges"].replace(" ", "0").astype(float)

## 2. Remove identifiers

`customerID` is a unique label with no predictive value, so it is removed.

In [2]:
df = df.drop(columns=["customerID"])
df.shape

(7043, 20)

## 3. Encode the target

`Churn` is converted to numbers: 1 = churned, 0 = stayed. The result matches the 27% churn baseline (1,869 of 7,043).

In [3]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

## 4. Separate features and target

- `X`: the 19 feature columns (inputs)
- `y`: the `Churn` column (what we predict)

In [4]:
X = df.drop(columns=["Churn"])

y = df["Churn"]

print(X.shape)
print(y.shape)

(7043, 19)
(7043,)


## 5. Train/test split

80% of customers are used for training (5,634) and 20% are held out for testing (1,409). The split is stratified, so both sets keep the 27% churn rate, and `random_state=42` makes it reproducible.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training customers:", len(X_train))
print("Test customers:", len(X_test))

Training customers: 5634
Test customers: 1409


## 6. Identify column types

15 text columns need encoding; 4 numeric columns need scaling. From here on, every preparation step is learned from the training set only, to prevent data leakage.

In [6]:
categorical_cols = X_train.select_dtypes(include="str").columns.tolist()
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()

print("Text columns:", len(categorical_cols))
print(categorical_cols)
print()
print("Number columns:", len(numeric_cols))
print(numeric_cols)

Text columns: 15
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Number columns: 4
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


## 7. One-hot encode text columns

The encoder is fit on the training set only, then applied to both sets. The 15 text columns become 36 binary (0/1) columns.

In [7]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)

encoder.fit(X_train[categorical_cols])

X_train_cat = encoder.transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

print(X_train_cat.shape)
print(X_test_cat.shape)

(5634, 36)
(1409, 36)


In [8]:
encoder.get_feature_names_out(categorical_cols)

array(['gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No', 'MultipleLines_No phone service',
       'MultipleLines_Yes', 'InternetService_DSL',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No', 'OnlineSecurity_No internet service',
       'OnlineSecurity_Yes', 'OnlineBackup_No',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No', 'DeviceProtection_No internet service',
       'DeviceProtection_Yes', 'TechSupport_No',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No', 'StreamingTV_No internet service',
       'StreamingTV_Yes', 'StreamingMovies_No',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_Month-to-month', 'Contract_One year',
       'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'Paymen

## 8. Scale numeric columns

The 4 numeric columns are standardized (0 = average, ±1 = one standard deviation). The scaler learns each column's average and spread from the training set only, then applies them to both sets.

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(X_train[numeric_cols])

X_train_num = scaler.transform(X_train[numeric_cols])
X_test_num = scaler.transform(X_test[numeric_cols])

pd.DataFrame({
    "Average": scaler.mean_,
    "Spread": scaler.scale_
}, index=numeric_cols).round(1)

,Average,Spread
SeniorCitizen,0.2,0.4
tenure,32.5,24.6
MonthlyCharges,64.9,30.1
TotalCharges,2299.3,2279.0


## 9. Combine features

Join the 36 encoded columns and 4 scaled columns into one table per set: 40 features in total.

In [10]:
import numpy as np

X_train_ready = np.hstack([X_train_num, X_train_cat])
X_test_ready = np.hstack([X_test_num, X_test_cat])

print(X_train_ready.shape)
print(X_test_ready.shape)

(5634, 40)
(1409, 40)


## Summary

The data is now fully numeric and ready for modeling:

| Set | Customers | Features |
|---|---|---|
| Training | 5,634 | 40 |
| Test | 1,409 | 40 |

- 15 text columns → 36 one-hot encoded columns
- 4 numeric columns → standardized
- All preparation steps were fit on the training set only, preventing data leakage.